## Fine Tuning Phi-4-mini-instruct

As there were some corrupted weights with Unsloth optimized model of this base version, we will be skipping Unsloth completely for fine tuning. We will use the standard **transformers** library from **Hugging Face** for fine tuning this model

In [4]:
!nvidia-smi

Tue May 19 19:37:08 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.90.12              Driver Version: 550.90.12      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX A6000               Off |   00000000:00:07.0 Off |                    0 |
| 30%   33C    P8             25W /  300W |   20119MiB /  46068MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
MODEL_NAME = "microsoft/Phi-4-mini-instruct"
DATASET_PATH = "/mnt/huggingface/data/medgemma_extracts"
OUTPUT_DIR = "/mnt/huggingface/models/phi-4-mini-instruct-ft"
MAX_SEQ_LENGTH = 4096
# NUM_TRAIN_EPOCHS = 3 # For dry run prefer using steps
BATCH_SIZE = 4
GRAD_ACCUM_STEPS = 8
LEARNING_RATE = 7e-5
SEED = 3407

## Inspecting System Prompt

Always inspect system prompt and check if it is properly formatted. A poorly formatted system prompt string leads to low accuracy of fine tuning and attention drift during inference

In [2]:
import json

JSON_SCHEMA = {
    "summary": "A concise, 1-2 sentence abstractive summary of the clinical scenario.",
    "clinical_reasoning": "A step-by-step logical breakdown of the diagnoses, treatments, or clinical decisions made in the text. Explain WHY certain relationships exist. Keep short and brief but to the point.",
    "relationships": [
        {
            "subject": "Source entity (e.g., Patient, Drug, Symptom)",
            "predicate": "Use STANDARD POSITIVE relationships (e.g., HAS_HISTORY, SHOWS_SYMPTOM, DIAGNOSED_WITH, PRESCRIBED). Do not use negated verbs like 'DENIES' or 'LACKS'.",
            "object": "Target entity",
            "polarity": "positive OR negative (Use 'negative' if the patient denies the history or lacks the symptom)",
            "certainty": "confirmed, suspected, OR hedged",
            "evidence": "The exact verbatim text snippet that proves this relationship."
        }
    ],
    "keywords": ["List", "of", "important", "clinical", "NER", "terms"]
}
SCHEMA_STRING = json.dumps(JSON_SCHEMA, indent=4)

SYSTEM_PROMPT = (
    "You are an expert clinical informatician. "
    f"Extract data strictly into this JSON schema:\n\n{SCHEMA_STRING}\n\n"
    "CRITICAL RULES:\n"
    "1. Use ONLY double quotes for all JSON keys and string values.\n"
    "2. Response MUST start with '{' and end with '}'.\n"
    "3. Output raw JSON only — no markdown, no code blocks.\n"
    "4. Provide values for ALL keys in the schema.\n"
    "5. Apostrophes in clinical terms (e.g., patient's) are allowed inside double-quoted strings.\n"
    "6. Extract at max 10 most clinically significant relationships only. "
    "Prioritize: diagnosis > treatment > symptoms > history.\n"
)

In [4]:
print(SYSTEM_PROMPT)

You are an expert clinical informatician. Extract data strictly into this JSON schema:

{
    "summary": "A concise, 1-2 sentence abstractive summary of the clinical scenario.",
    "clinical_reasoning": "A step-by-step logical breakdown of the diagnoses, treatments, or clinical decisions made in the text. Explain WHY certain relationships exist. Keep short and brief but to the point.",
    "relationships": [
        {
            "subject": "Source entity (e.g., Patient, Drug, Symptom)",
            "predicate": "Use STANDARD POSITIVE relationships (e.g., HAS_HISTORY, SHOWS_SYMPTOM, DIAGNOSED_WITH, PRESCRIBED). Do not use negated verbs like 'DENIES' or 'LACKS'.",
            "object": "Target entity",
            "polarity": "positive OR negative (Use 'negative' if the patient denies the history or lacks the symptom)",
            "certainty": "confirmed, suspected, OR hedged",
            "evidence": "The exact verbatim text snippet that proves this relationship."
        }
    ],
  

## Load Model And Tokenizer

In [6]:
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Load and configure tokenizer padding
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Create bits and bytes configuration for n-bit fine tuning
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

# Load the model with bits and bytes config
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    attn_implementation="flash_attention_2",
    dtype=torch.bfloat16
)

# Required for 4-bit training
model = prepare_model_for_kbit_training(model)
model.config.use_cache = False
model.config.pretraining_tp = 1 # keep 1 for single GPU training (no distribution)

# LoRA configuration
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.0,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ]
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.


Loading weights:   0%|          | 0/194 [00:00<?, ?it/s]

trainable params: 8,912,896 || all params: 3,844,934,656 || trainable%: 0.2318


## Model Insepction

In [ ]:
# Print all linear layer names - very important
# Need to know exact names of linear layers for fine tuning
for name, module in model.named_modules():
    if isinstance(module, torch.nn.Linear):
        print(name)

base_model.model.model.layers.0.self_attn.o_proj.base_layer
base_model.model.model.layers.0.self_attn.o_proj.lora_A.default
base_model.model.model.layers.0.self_attn.o_proj.lora_B.default
base_model.model.model.layers.0.self_attn.qkv_proj
base_model.model.model.layers.0.mlp.gate_up_proj
base_model.model.model.layers.0.mlp.down_proj.base_layer
base_model.model.model.layers.0.mlp.down_proj.lora_A.default
base_model.model.model.layers.0.mlp.down_proj.lora_B.default
base_model.model.model.layers.1.self_attn.o_proj.base_layer
base_model.model.model.layers.1.self_attn.o_proj.lora_A.default
base_model.model.model.layers.1.self_attn.o_proj.lora_B.default
base_model.model.model.layers.1.self_attn.qkv_proj
base_model.model.model.layers.1.mlp.gate_up_proj
base_model.model.model.layers.1.mlp.down_proj.base_layer
base_model.model.model.layers.1.mlp.down_proj.lora_A.default
base_model.model.model.layers.1.mlp.down_proj.lora_B.default
base_model.model.model.layers.2.self_attn.o_proj.base_layer
base_m

In [ ]:
# RELOAD base model first to avoid double-wrapping
model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Phi-4-mini-instruct",
    torch_dtype="auto",
    device_map="auto"
)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.0,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "qkv_proj",      # Fused Q+K+V (replaces q_proj, k_proj, v_proj)
        "o_proj",        # Attention output (you had this correct)
        "gate_up_proj",  # Fused gate+up MLP (replaces gate_proj, up_proj)
        "down_proj",     # MLP down (you had this correct)
    ],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

Loading weights:   0%|          | 0/194 [00:00<?, ?it/s]

trainable params: 23,068,672 || all params: 3,859,090,432 || trainable%: 0.5978


: 

## Dataset Preparation - Dry Run

Performing a dry run before fine tuning is very important. It allows us to analyze the fine tuning process, select the best hyperparameters and evaluate inference. For dry runs, we can use the train set and break it into train and eval sets. 

Since this is a dry run, 150-180 records are more than enough for train set and 30-50 records enough for eval set. Shuffle the train set, then split it into train-test with 90-10 split and finally select a subset of 150 and 40 records from each subset.  

Once the final data slices are prepared, map over them with a transform function that transforms the raw records into a fine tuning recognized format

In [7]:
from datasets import load_from_disk

# Only load the train set for now
# Use a slice of train set for dry run
dataset = load_from_disk(f"{DATASET_PATH}/train")

def format_example_new(example):
    """
    Format columns into standard Hugging Face conversation dictionaries.
    The output dict MUST expose a single structural column names "messages"
    """
    # Ground truth from medgemma extractions
    target_json = json.dumps(example["data"], indent=2)

     # Build prompt string with chat template (includes assistant header)
    prompt_messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"CONTEXT:\n{example['raw_medical_text']}"}
    ]
    prompt_text = tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=False,
        add_generation_prompt=True  # Adds the assistant role header (e.g., <|assistant|>)
    )

    # Completion is just the JSON output + EOS
    completion_text = target_json + tokenizer.eos_token

    return {
        "prompt": prompt_text,      
        "completion": completion_text  
    }

def format_example(example):
    """
    Convert a single patient record into a chat-formatted training text
    """
    # Ground truth from medgemma extractions
    target_json = json.dumps(example["data"], indent=2)

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"CONTEXT:\n{example['raw_medical_text']}"},
        {"role": "assistant", "content": target_json}
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )
    return {"text": text}

def tokenize_and_mask(example):
    """
    Mask everything before <|assistant|> token
    Model only learns from assistant response tokens
    """
    tokenized = tokenizer(
        example["text"],
        truncation=True,
        max_length=4096,
        padding=False
    )

    input_ids = tokenized["input_ids"]

    # Find where assistant response starts
    assistant_ids = tokenizer.encode(
        "<|assistant|>",
        add_special_tokens=False
    )

    assistant_pos = None
    for i in range(len(input_ids) - len(assistant_ids)):
        if input_ids[i:i + len(assistant_ids)] == assistant_ids:
            assistant_pos = i
            break

    if assistant_pos is None:
        # <|assistant|> token was truncated away entirely
        # Return empty labels to signal this record should be skipped
        tokenized["labels"] = [-100] * len(input_ids)
        tokenized["valid"] = False
        return tokenized

    labels = [-100] * len(input_ids)
    labels[assistant_pos + len(assistant_ids):] = input_ids[assistant_pos + len(assistant_ids):]

    tokenized["labels"] = labels
    tokenized["valid"] = True 
    
    return tokenized

# Randomize the train dataset 
# Split entire train set into smaller train and test
train = dataset.shuffle(seed=42) 
split = train.train_test_split(test_size=0.1, seed=42)

# Use 150 random records for training and 20 random records for evaluation
train_split = split["train"].select(range(200))
eval_split = split["test"].select(range(50))

# Format into chat template
train_formatted = train_split.map(
    format_example_new, # TODO: Finalize the mapping function
    desc="Formatting train"
)

# TODO: Update as above
# eval_formatted = eval_split.map(
#     format_example,
#     desc="Formatting eval"
# )

# TODO: Verify requirement
# Tokenize and mask instructions
# train_dataset = train_formatted.map(
#     tokenize_and_mask,
#     remove_columns=train_formatted.column_names,
#     desc="Masking train"
# ).filter(lambda x: x["valid"])

# eval_dataset = eval_formatted.map(
#     tokenize_and_mask,
#     remove_columns=eval_formatted.column_names,
#     desc="Masking eval"
# ).filter(lambda x: x["valid"])

Formatting train:   0%|          | 0/200 [00:00<?, ? examples/s]

In [8]:
# Always inspect final dataset before fine tuning run
sample = train_formatted[0]
print("--- PROMPT ---")
print(sample["prompt"])
print("\n--- COMPLETION ---")
print(sample["completion"])

--- PROMPT ---
<|system|>You are an expert clinical informatician. Extract data strictly into this JSON schema:

{
    "summary": "A concise, 1-2 sentence abstractive summary of the clinical scenario.",
    "clinical_reasoning": "A step-by-step logical breakdown of the diagnoses, treatments, or clinical decisions made in the text. Explain WHY certain relationships exist. Keep short and brief but to the point.",
    "relationships": [
        {
            "subject": "Source entity (e.g., Patient, Drug, Symptom)",
            "predicate": "Use STANDARD POSITIVE relationships (e.g., HAS_HISTORY, SHOWS_SYMPTOM, DIAGNOSED_WITH, PRESCRIBED). Do not use negated verbs like 'DENIES' or 'LACKS'.",
            "object": "Target entity",
            "polarity": "positive OR negative (Use 'negative' if the patient denies the history or lacks the symptom)",
            "certainty": "confirmed, suspected, OR hedged",
            "evidence": "The exact verbatim text snippet that proves this relations

## WandB Setup

It is very important to log results to wandb. This allows us to monitor the training runs, watch for overfitting etc. 

In [41]:
import wandb
import json
from dotenv import load_dotenv
from transformers import TrainerCallback

load_dotenv()

wandb.init(
    project="phi-4-mini-instruct-ft",
    name="dry-run-lr7e5-r16",
    config={
        "model": "phi4-mini-instruct",
        "r": 16,
        "lora_alpha": 32,
        "learning_rate": 7e-5,
        "steps": 150,
        "batch_size": 4,
        "grad_accum": 8,
        "train_records": 200,
        "eval_records": 50
    }
)           

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: loknezmonzter (loknezmonzter-dev) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: Detected [huggingface_hub.inference] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai


## Fine Tuning Configuration

In [42]:
from trl import SFTTrainer, SFTConfig

# Training configuration for fine tuning
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,

    # --- Set step count for train --- #
    num_train_epochs=-1,
    max_steps=150, # ensures enough optimizer steps to give a clear picture of dry run

    # --- Logging and eval --- #
    logging_steps=5,
    eval_steps=25,
    eval_strategy="steps",
    save_strategy="steps",
    save_steps=50,
    report_to="wandb",

    # --- Batch configuration --- #
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},

    # --- Optimizations --- #
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    weight_decay=0.01,
    max_grad_norm=1.0,
    optim="paged_adamw_8bit",

    # --- Precision control --- #
    bf16=True,
    tf32=True,

    # --- Dataset options --- #
    # max_length=MAX_SEQ_LENGTH, # not needed -> handled in tokenize_and_mask
    # dataset_text_field="text", # not needed -> dataset has no text column anymore
    packing=False,
   
    seed=SEED
)

# Initialize trainer
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    peft_config=lora_config
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
/workspace/.venv/lib/python3.11/site-packages/peft/tuners/lora/bnb.py:373: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(


In [44]:
try:
    trainer_stats = trainer.train()
finally:
    wandb.finish()

Casting fp32 inputs back to torch.bfloat16 for flash-attn compatibility.


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
25,0.408278,0.479117,1.562121,744436.000000,0.851718
50,0.329854,0.433981,1.464278,1464320.000000,0.865624
75,0.260351,0.431904,1.370385,2208991.000000,0.865339
100,0.226040,0.446872,1.292517,2929110.000000,0.866242
125,0.197994,0.466373,1.258023,3673546.000000,0.864252
150,0.183769,0.468675,1.253022,4393641.000000,0.865901


eval/entropy,█▆▄▂▁▁
eval/loss,█▁▁▃▆▆
eval/mean_token_accuracy,▁███▇█
eval/num_tokens,▁▂▄▅▇█
eval/runtime,█▁▂▄▁▂
eval/samples_per_second,▁█▇▅█▇
eval/steps_per_second,▁█▇▆▇▇
train/entropy,▇█▇██▆▆▆▆▅▅▅▄▄▃▃▃▂▂▃▂▂▂▂▂▁▂▁▂▁
train/epoch,▁▁▁▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇████
train/global_step,▁▁▁▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▆▆▆▆▆▇▇▇▇▇█████
+5,...


In [1]:
import trl
trl.__version__

'0.24.0'